In [1]:
#setup + imports

import sys
sys.path.append('..')

import pandas as pd
import os
import re
from config import engine

# Helper function to extract uid from filename
def extract_uid(filename):
    """Extract uid from filenames like 'activity_u01.csv' -> 'u01'"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")

✓ Setup complete


In [2]:
# ===========================
# COMPREHENSIVE DATA LOADING VERIFICATION
# ===========================

print("="*70)
print("DATABASE POPULATION STATUS - FULL CHECK")
print("="*70)

# Expected student count
expected_students = 49

# All tables organized by category
tables_to_check = {
    'SPINE': ['student'],
    'SENSING': ['activity_reading', 'audio_reading', 'gps_reading', 'bluetooth_scan', 
                'conversation', 'dark_period', 'phone_lock', 'phone_charge', 
                'wifi_scan', 'wifi_location'],
    'BEHAVIORAL': ['sms', 'call_log', 'app_usage', 'calendar', 'dinning'],
    'EMA': ['ema_definition', 'ema_response'],
    'SURVEY': ['phq9_response', 'pss_response', 'panas_response', 'bigfive_response',
               'flourishing_response', 'loneliness_response', 'psqi_response', 'vr12_response'],
    'EDUCATION': ['grade', 'deadline', 'piazza_usage', 'class_info', 'enrollment']
}

for category, tables in tables_to_check.items():
    print(f"\n{'='*70}")
    print(f"{category} TABLES")
    print('='*70)
    
    for table in tables:
        try:
            # Total row count
            result = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", engine)
            total_rows = result['count'][0]
            
            # Student count (if uid column exists)
            try:
                result = pd.read_sql(f"SELECT COUNT(DISTINCT uid) as students FROM {table}", engine)
                student_count = result['students'][0]
                coverage = (student_count / expected_students) * 100
                
                if student_count == expected_students:
                    status = "✅"
                elif student_count > 0:
                    status = f"⚠️ {student_count}/{expected_students}"
                else:
                    status = "⚠️ EMPTY"
                    
                print(f"{status} {table:25s}: {total_rows:>12,} rows | {student_count:>2} students ({coverage:.0f}%)")
            except:
                # No uid column (reference tables like ema_definition, class_info)
                if total_rows > 0:
                    status = "✅"
                    print(f"{status} {table:25s}: {total_rows:>12,} rows | (reference table)")
                else:
                    status = "⚠️ EMPTY"
                    print(f"{status} {table:25s}: {total_rows:>12,} rows | (reference table)")
                
        except Exception as e:
            print(f"❌ {table:25s}: Table doesn't exist")

print("\n" + "="*70)
print("VERIFICATION COMPLETE")
print("="*70)

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
total_tables = sum(len(tables) for tables in tables_to_check.values())
print(f"Total tables in schema: {total_tables}")
print(f"Expected students: {expected_students}")
print("\nCheck for ⚠️ or ❌ markers above to see what's missing!")
print("="*70)

DATABASE POPULATION STATUS - FULL CHECK

SPINE TABLES
✅ student                  :           49 rows | 49 students (100%)

SENSING TABLES
✅ activity_reading         :   22,842,191 rows | 49 students (100%)
⚠️ 1/49 audio_reading            :      381,000 rows |  1 students (2%)
✅ gps_reading              :      202,877 rows | 49 students (100%)
✅ bluetooth_scan           :    1,288,526 rows | 49 students (100%)
✅ conversation             :       79,023 rows | 49 students (100%)
✅ dark_period              :        7,269 rows | 49 students (100%)
✅ phone_lock               :        9,275 rows | 49 students (100%)
✅ phone_charge             :        3,318 rows | 49 students (100%)
✅ wifi_scan                :   19,240,217 rows | 49 students (100%)
✅ wifi_location            :    3,787,676 rows | 49 students (100%)

BEHAVIORAL TABLES
⚠️ EMPTY sms                      :            0 rows |  0 students (0%)
✅ call_log                 :       71,801 rows | 49 students (100%)
✅ app_usage       